In [1]:
import pandas as pd
import requests
import plotly.express as px

In [2]:
lat = 40.7178
lng = -74.0431

start_date = "2025-01-01"
end_date = "2025-12-31"

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat,
    "longitude": lng,
    "start_date": start_date,
    "end_date": end_date,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

weather_daily = pd.DataFrame(data["daily"])
weather_daily["date"] = pd.to_datetime(weather_daily["time"])
weather_daily = weather_daily.drop(columns="time")

weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


### Observe Missing Values

In [3]:
weather_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   temperature_2m_max   365 non-null    float64       
 1   temperature_2m_min   365 non-null    float64       
 2   temperature_2m_mean  365 non-null    float64       
 3   precipitation_sum    365 non-null    float64       
 4   rain_sum             365 non-null    float64       
 5   snowfall_sum         365 non-null    float64       
 6   wind_speed_10m_max   365 non-null    float64       
 7   date                 365 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(7)
memory usage: 22.9 KB


In [4]:
missing_values = (
    weather_daily
    .isna()
    .sum()
    .reset_index())

missing_values.columns = ['column', 'missing']
missing_values

,column,missing
0,temperature_2m_max,0
1,temperature_2m_min,0
2,temperature_2m_mean,0
3,precipitation_sum,0
4,rain_sum,0
5,snowfall_sum,0
6,wind_speed_10m_max,0
7,date,0


### Store the data

In [5]:
OUTPUT_DIR = "../data/JC/"
weather_daily.to_csv(f'{OUTPUT_DIR}/jersey_weather_2025.csv', index = False)

### Weather Data Visualizations

In [6]:
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.dtypes

temperature_2m_max            float64
temperature_2m_min            float64
temperature_2m_mean           float64
precipitation_sum             float64
rain_sum                      float64
snowfall_sum                  float64
wind_speed_10m_max            float64
date                   datetime64[us]
dtype: object

### Temperature

In [7]:
fig = px.line(
    weather_daily,
    x="date",
    y="temperature_2m_mean",
    title="Daily Average Temperature Over Time",
    markers=False
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Average Temperature",
    hovermode="x unified"
)

fig.show()

In [ ]:
temperature_long = weather_daily.melt(
    id_vars="date",
    value_vars=[
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean"
    ],
    var_name="temperature_type",
    value_name="temperature"
)

temperature_long.head()

In [ ]:
fig = px.line(
    temperature_long,
    x="date",
    y="temperature",
    color="temperature_type",
    title="Daily Temperature: Maximum, Minimum, and Average",
    markers=False
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Temperature",
    hovermode="x unified"
)

fig.show()

### Wind

In [ ]:
fig = px.line(
    weather_daily,
    x="date",
    y="wind_speed_10m_max",
    title="Daily Maximum Wind Speed Over Time",
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Maximum Wind Speed",
    hovermode="x unified"
)

fig.show()